IMPORT LIBRARY

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import json
import pickle
import random
import numpy as np
import tensorflow as tf

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.regularizers import l2

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


SET SEED

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

LOAD DATASET

In [3]:
dataset_path = "../data/clean_recipes_5000.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    recipes = json.load(f)

print(f"Jumlah dataset loaded: {len(recipes)}")

Jumlah dataset loaded: 5000


In [4]:
recipe_texts = []
categories = []

for recipe in recipes:
    ingredients = recipe.get("Ingredients Cleaned", "")
    category = recipe.get("Category", "unknown")

    if (
        ingredients
        and isinstance(ingredients, str)
        and ingredients.strip()
    ):
        recipe_texts.append(ingredients.lower())
        categories.append(category.lower())

print(f"Jumlah resep valid: {len(recipe_texts)}")

Jumlah resep valid: 5000


TF-IDF VECTORIZATION

In [5]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2
)

X = vectorizer.fit_transform(recipe_texts)
X = X.toarray().astype(np.float32)

print(f"Bentuk TF-IDF Matrix: {X.shape}")

Bentuk TF-IDF Matrix: (5000, 1685)


LABEL ENCODING

In [6]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(categories)

print(f"Jumlah kelas kategori: {len(label_encoder.classes_)}")
print(label_encoder.classes_)

Jumlah kelas kategori: 8
['ayam' 'ikan' 'kambing' 'sapi' 'tahu' 'telur' 'tempe' 'udang']


SPLIT DATASET

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp
)

print(f"Jumlah data training: {len(X_train)}")
print(f"Jumlah data validation: {len(X_val)}")
print(f"Jumlah data testing: {len(X_test)}")

Jumlah data training: 3500
Jumlah data validation: 750
Jumlah data testing: 750


CUSTOM LAYER

In [8]:
@register_keras_serializable()
class IngredientsImportanceLayer(Layer):

    def __init__(self, factor=1.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs):
        return inputs * self.factor

    def get_config(self):
        config = super().get_config()
        config.update({
            "factor": self.factor
        })
        return config

CUSTOM LOSS FUNCTION

In [9]:
@register_keras_serializable()
def custom_recipe_loss(y_true, y_pred):
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true,
        y_pred
    )
    return tf.reduce_mean(loss)

CUSTOM CALLBACK

In [10]:
class TrainingLogger(Callback):

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        print(
            f"\nEpoch {epoch + 1} selesai | "
            f"Loss: {logs.get('loss', 0):.4f} | "
            f"Accuracy: {logs.get('accuracy', 0):.4f}"
        )

BUILD MODEL

In [11]:
def build_mlp_model(input_dim=5000, num_classes=8):
    input_layer = Input(shape=(input_dim,))

    x = IngredientsImportanceLayer()(input_layer)

    x = Dense(
        128,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = Dropout(0.4)(x)

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = Dropout(0.4)(x)

    output_layer = Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = Model(
        inputs=input_layer,
        outputs=output_layer
    )

    model.compile(
        optimizer="adam",
        loss=custom_recipe_loss,
        metrics=["accuracy"]
    )

    return model


model = build_mlp_model(
    input_dim=X.shape[1],
    num_classes=len(label_encoder.classes_)
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1685)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ingredients_importance_layer    │ (None, 1685)           │             0 │
│ (IngredientsImportanceLayer)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       215,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,584 (877.28 KB)

 Trainable params: 224,584 (877.28 KB)

 Non-trainable params: 0 (0.00 B)

DATASET TRAINING & TENSORBOARD

In [12]:
optimizer = tf.keras.optimizers.Adam()

batch_size = 32
epochs = 5

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(1000, seed=SEED)
    .batch(batch_size)
)

log_dir = "../logs/classification"
writer = tf.summary.create_file_writer(log_dir)

print(f"TensorBoard logs: {log_dir}")

TensorBoard logs: ../logs/classification


TRAINING CUSTOM LOOP & GRADIENTTAPE

In [13]:
for epoch in range(epochs):
    epoch_losses = []
    epoch_accs = []

    for batch_x, batch_y in train_dataset:
        with tf.GradientTape() as tape:
            predictions = model(
                batch_x,
                training=True
            )

            loss = custom_recipe_loss(
                batch_y,
                predictions
            )

        grads = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(
                grads,
                model.trainable_variables
            )
        )

        pred_class = tf.argmax(
            predictions,
            axis=1
        )

        acc = tf.reduce_mean(
            tf.cast(
                pred_class == batch_y,
                tf.float32
            )
        )

        epoch_losses.append(loss.numpy())
        epoch_accs.append(acc.numpy())

    avg_loss = np.mean(epoch_losses)
    avg_acc = np.mean(epoch_accs)

    val_predictions = model(
        X_val,
        training=False
    )

    val_pred_class = np.argmax(
        val_predictions,
        axis=1
    )

    val_acc = accuracy_score(
        y_val,
        val_pred_class
    )

    val_loss = custom_recipe_loss(
        y_val,
        val_predictions
    ).numpy()

    print(
        f"Epoch {epoch + 1}/{epochs}"
        f" | Loss={avg_loss:.4f}"
        f" | Accuracy={avg_acc:.4f}"
        f" | Val Loss={val_loss:.4f}"
        f" | Val Accuracy={val_acc:.4f}"
    )

    with writer.as_default():
        tf.summary.scalar("loss", avg_loss, step=epoch)
        tf.summary.scalar("accuracy", avg_acc, step=epoch)
        tf.summary.scalar("val_loss", val_loss, step=epoch)
        tf.summary.scalar("val_accuracy", val_acc, step=epoch)

Epoch 1/5 | Loss=1.9273 | Accuracy=0.3394 | Val Loss=1.4940 | Val Accuracy=0.7213
Epoch 2/5 | Loss=1.0886 | Accuracy=0.6995 | Val Loss=0.6537 | Val Accuracy=0.8600
Epoch 3/5 | Loss=0.5380 | Accuracy=0.8659 | Val Loss=0.4176 | Val Accuracy=0.8813
Epoch 4/5 | Loss=0.3245 | Accuracy=0.9180 | Val Loss=0.3699 | Val Accuracy=0.8880
Epoch 5/5 | Loss=0.2292 | Accuracy=0.9433 | Val Loss=0.3522 | Val Accuracy=0.8880


EVALUASI DATA TEST

In [14]:
test_predictions = model(
    X_test,
    training=False
)

test_pred_class = np.argmax(
    test_predictions,
    axis=1
)

test_accuracy = accuracy_score(
    y_test,
    test_pred_class
)

test_loss = custom_recipe_loss(
    y_test,
    test_predictions
).numpy()

print("\nHASIL EVALUASI")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")

if test_accuracy >= 0.85:
    print("Status Accuracy: MEMENUHI syarat accuracy minimal 85%")
else:
    print("Status Accuracy: BELUM memenuhi syarat accuracy minimal 85%")


HASIL EVALUASI
Loss: 0.3136
Accuracy: 0.9133
Status Accuracy: MEMENUHI syarat accuracy minimal 85%


SAVE MODEL

In [15]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(
    models_dir,
    "mlp_recipe_model.keras"
)

model.save(model_path)

vectorizer_path = os.path.join(
    models_dir,
    "tfidf_vectorizer.pkl"
)

with open(vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)

label_encoder_path = os.path.join(
    models_dir,
    "label_encoder.pkl"
)

with open(label_encoder_path, "wb") as f:
    pickle.dump(label_encoder, f)

writer.flush()
writer.close()

print(f"Model disimpan: {model_path}")
print(f"Vectorizer disimpan: {vectorizer_path}")
print(f"Label Encoder disimpan: {label_encoder_path}")
print(f"TensorBoard logs: {log_dir}")

Model disimpan: ../models\mlp_recipe_model.keras
Vectorizer disimpan: ../models\tfidf_vectorizer.pkl
Label Encoder disimpan: ../models\label_encoder.pkl
TensorBoard logs: ../logs/classification


In [1]:
"""
Jalankan TensorBoard dengan perintah berikut di terminal:

cd "quest\\Path AI"
tensorboard --logdir logs --port 6006

Lalu buka:
http://localhost:6006

Catatan:
- Grafik classification berasal dari logs/classification
- Grafik quality_score berasal dari logs/quality_score
"""

'\nJalankan TensorBoard dengan perintah berikut di terminal:\n\ncd "quest\\Path AI"\ntensorboard --logdir logs --port 6006\n\nLalu buka:\nhttp://localhost:6006\n\nCatatan:\n- Grafik classification berasal dari logs/classification\n- Grafik quality_score berasal dari logs/quality_score\n'